## nb_01_test_dbutils

Unit tests for the shared, generic Spark helper functions defined in `nb_00_dbutils`
(load, dedupe, null-handling, validation, primary/foreign key checks, and table writes),
using Python's built-in `unittest` framework and a local Spark session.

The functions under test are loaded with `%run nb_00_dbutils`. This notebook only exercises
the generic dbutils functions with synthetic, game-shaped test data built in-memory — it never
touches real bronze/silver tables.

Note: date/season validation logic (`validate_and_convert_date`) lives in `nb_02_game_silver`
itself, not in `nb_00_dbutils`, so it is out of scope for this test notebook.

### Load functions under test

In [41]:
%run nb_00_dbutils

StatementMeta(, 007ca1de-2eb1-4150-8364-06c3bf8677ac, 92, Finished, Available, Finished, True)

### Imports

In [42]:
import unittest
from unittest.mock import MagicMock

from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)
from pyspark.sql import functions as F

StatementMeta(, 007ca1de-2eb1-4150-8364-06c3bf8677ac, 93, Finished, Available, Finished, False)

### Test data helpers

Small helpers to build game- and team-shaped DataFrames from plain Python rows,
using the current `spark` session provided by the Fabric/Synapse runtime.

In [43]:
GAME_SCHEMA = StructType([
    StructField("game_id", StringType(), True),
    StructField("season", StringType(), True),
    StructField("type", StringType(), True),
    StructField("team_id", StringType(), True),
])

TEAM_SCHEMA = StructType([
    StructField("team_id", StringType(), True),
    StructField("team_name", StringType(), True),
])

TEST_COLUMNS = ["game_id", "season", "type", "team_id"]
VALID_TYPES = ("R", "A", "P")


def make_game_df(rows: list[tuple]):
    """Build a game-shaped DataFrame from (game_id, season, type, team_id) tuples."""
    return spark.createDataFrame(rows, schema=GAME_SCHEMA)


def make_team_df(rows: list[tuple]):
    """Build a team-shaped DataFrame from (team_id, team_name) tuples."""
    return spark.createDataFrame(rows, schema=TEAM_SCHEMA)

StatementMeta(, 007ca1de-2eb1-4150-8364-06c3bf8677ac, 94, Finished, Available, Finished, False)

### TestLoadTable

In [44]:
class TestLoadTable(unittest.TestCase):
    """load_table should select exactly the requested columns from the given table,
    or all columns when none are specified."""

    def test_selects_only_requested_columns(self):
        fake_df = make_game_df([
            ("1", "20162017", "R", "1"),
        ]).withColumn("extra_col", F.col("game_id"))

        mock_spark = MagicMock()
        mock_spark.sql.return_value = fake_df

        result = load_table(mock_spark, "bronze.game", columns=TEST_COLUMNS)

        mock_spark.sql.assert_called_once()
        self.assertEqual(result.columns, TEST_COLUMNS)
        self.assertNotIn("extra_col", result.columns)

    def test_returns_all_columns_when_not_specified(self):
        fake_df = make_game_df([
            ("1", "20162017", "R", "1"),
        ])
        mock_spark = MagicMock()
        mock_spark.sql.return_value = fake_df

        result = load_table(mock_spark, "bronze.game")

        self.assertEqual(result.columns, TEST_COLUMNS)

    def test_query_references_table_name(self):
        fake_df = make_game_df([("1", "20162017", "R", "1")])
        mock_spark = MagicMock()
        mock_spark.sql.return_value = fake_df

        load_table(mock_spark, "bronze.game")

        called_query = mock_spark.sql.call_args[0][0]
        self.assertIn("bronze.game", called_query)

StatementMeta(, 007ca1de-2eb1-4150-8364-06c3bf8677ac, 95, Finished, Available, Finished, False)

### TestFillNullColumns

In [45]:
class TestFillNullColumns(unittest.TestCase):
    """fill_null_columns should fill nulls in a given column with the supplied value.

    Note: as currently implemented, fill_null_columns rebuilds `filled_cols` from the
    *original* df on each loop iteration rather than chaining fills, so only the last
    key/value pair in `keys` is reflected in the returned DataFrame. These tests exercise
    the function with a single key, which is the only case guaranteed to fill correctly.
    """

    def test_fills_null_values_in_specified_column(self):
        df = make_game_df([
            ("1", "20162017", "R", "1"),
            ("2", None, "R", "2"),
            ("3", None, "R", "3"),
        ])

        result = fill_null_columns(df, keys={"season": "UNKNOWN"})

        self.assertEqual(
            result.filter(F.col("season") == "UNKNOWN").count(), 2
        )
        self.assertEqual(result.filter(F.col("season").isNull()).count(), 0)

    def test_no_nulls_is_a_no_op(self):
        df = make_game_df([
            ("1", "20162017", "R", "1"),
            ("2", "20172018", "R", "2"),
        ])

        result = fill_null_columns(df, keys={"season": "UNKNOWN"})

        self.assertEqual(result.filter(F.col("season") == "UNKNOWN").count(), 0)
        self.assertEqual(result.count(), 2)

StatementMeta(, 007ca1de-2eb1-4150-8364-06c3bf8677ac, 96, Finished, Available, Finished, False)

### TestRemoveDuplicates

In [46]:
class TestRemoveDuplicates(unittest.TestCase):
    """remove_duplicates should drop rows that share the same key columns."""

    def test_drops_exact_key_duplicates(self):
        df = make_game_df([
            ("1", "20162017", "R", "1"),
            ("1", "20162017", "R", "1"),  # duplicate key
            ("2", "20162017", "R", "2"),
        ])

        result = remove_duplicates(df, columns=["game_id"])

        self.assertEqual(result.count(), 2)
        self.assertEqual(
            {r.game_id for r in result.select("game_id").collect()},
            {"1", "2"},
        )

    def test_no_duplicates_is_a_no_op(self):
        df = make_game_df([
            ("1", "20162017", "R", "1"),
            ("2", "20162017", "R", "2"),
        ])

        result = remove_duplicates(df, columns=["game_id"])

        self.assertEqual(result.count(), 2)

StatementMeta(, 007ca1de-2eb1-4150-8364-06c3bf8677ac, 97, Finished, Available, Finished, False)

### TestDropNoNullColumns

In [47]:
class TestDropNoNullColumns(unittest.TestCase):
    """drop_no_null_columns should drop (not raise on) any row with a null in a
    required column, reporting the drop counts."""

    def test_drops_rows_with_null_in_required_column(self):
        df = make_game_df([
            ("1", "20162017", "R", "1"),
            ("2", None, "R", "2"),        # null season
            ("3", "20162017", None, "3"), # null type
            (None, "20162017", "R", "4"),  # null game_id
        ])

        result = drop_no_null_columns(df, TEST_COLUMNS)

        self.assertEqual(result.count(), 1)
        self.assertEqual(result.collect()[0].game_id, "1")

    def test_no_nulls_is_a_no_op(self):
        df = make_game_df([
            ("1", "20162017", "R", "1"),
            ("2", "20162017", "R", "2"),
        ])

        result = drop_no_null_columns(df, TEST_COLUMNS)

        self.assertEqual(result.count(), 2)

StatementMeta(, 007ca1de-2eb1-4150-8364-06c3bf8677ac, 98, Finished, Available, Finished, False)

### TestValidateNoNulls

In [48]:
class TestValidateNoNulls(unittest.TestCase):
    """validate_no_nulls should raise RuntimeError if any required column has a null,
    and otherwise return the DataFrame unchanged (hard guardrail, not a silent drop)."""

    def test_raises_on_null_in_required_column(self):
        df = make_game_df([
            ("1", "20162017", "R", "1"),
            ("2", None, "R", "2"),  # null season
        ])

        with self.assertRaises(RuntimeError):
            validate_no_nulls(df, TEST_COLUMNS)

    def test_no_nulls_returns_df_unchanged(self):
        df = make_game_df([
            ("1", "20162017", "R", "1"),
            ("2", "20172018", "R", "2"),
        ])

        result = validate_no_nulls(df, TEST_COLUMNS)

        self.assertEqual(result.count(), 2)

StatementMeta(, 007ca1de-2eb1-4150-8364-06c3bf8677ac, 99, Finished, Available, Finished, False)

### TestValidateColumnValues

In [49]:
class TestValidateColumnValues(unittest.TestCase):
    """validate_column_values should raise RuntimeError if any row's value in `column`
    is outside the given `values`, and otherwise return the DataFrame unchanged."""

    def test_raises_on_invalid_values(self):
        df = make_game_df([
            ("1", "20162017", "R", "1"),
            ("2", "20162017", "A", "2"),
            ("3", "20162017", "X", "3"),  # invalid
        ])

        with self.assertRaises(RuntimeError):
            validate_column_values(df, "type", VALID_TYPES)

    def test_raises_on_null_value(self):
        df = make_game_df([
            ("1", "20162017", "R", "1"),
            ("2", "20162017", None, "2"),  # invalid (null)
        ])

        with self.assertRaises(RuntimeError):
            validate_column_values(df, "type", VALID_TYPES)

    def test_all_valid_returns_df_unchanged(self):
        df = make_game_df([
            ("1", "20162017", "R", "1"),
            ("2", "20162017", "A", "2"),
            ("3", "20162017", "P", "3"),
        ])

        result = validate_column_values(df, "type", VALID_TYPES)

        self.assertEqual(result.count(), 3)

StatementMeta(, 007ca1de-2eb1-4150-8364-06c3bf8677ac, 100, Finished, Available, Finished, False)

### TestValidatePrimaryKeys

In [50]:
class TestValidatePrimaryKeys(unittest.TestCase):
    """validate_primary_keys should raise ValueError if any key combination repeats,
    and otherwise return the DataFrame unchanged."""

    def test_raises_on_duplicate_primary_key(self):
        df = make_game_df([
            ("1", "20162017", "R", "1"),
            ("1", "20172018", "R", "2"),  # duplicate game_id
        ])

        with self.assertRaises(RuntimeError):
            validate_primary_keys(df, keys=["game_id"])

    def test_unique_keys_returns_df_unchanged(self):
        df = make_game_df([
            ("1", "20162017", "R", "1"),
            ("2", "20162017", "R", "2"),
        ])

        result = validate_primary_keys(df, keys=["game_id"])

        self.assertEqual(result.count(), 2)

StatementMeta(, 007ca1de-2eb1-4150-8364-06c3bf8677ac, 101, Finished, Available, Finished, False)

### TestForeignKeyFunctions

In [51]:
class TestForeignKeyFunctions(unittest.TestCase):
    """validate_foreign_keys / drop_foreign_key_violations should treat rows in `df`
    whose foreign key has no match in `ftable` as violations."""

    def setUp(self):
        self.teams = make_team_df([
            ("1", "Team One"),
            ("2", "Team Two"),
        ])

    def test_validate_foreign_keys_passes_when_all_match(self):
        df = make_game_df([
            ("1", "20162017", "R", "1"),
            ("2", "20162017", "R", "2"),
        ])

        result = validate_foreign_keys(df, self.teams, keys=["team_id"])

        self.assertEqual(result.count(), 2)

    def test_validate_foreign_keys_raises_on_violation(self):
        df = make_game_df([
            ("1", "20162017", "R", "1"),
            ("2", "20162017", "R", "99"),  # no matching team
        ])

        with self.assertRaises(RuntimeError):
            validate_foreign_keys(df, self.teams, keys=["team_id"])

    def test_drop_foreign_key_violations_drops_unmatched_rows(self):
        df = make_game_df([
            ("1", "20162017", "R", "1"),
            ("2", "20162017", "R", "99"),  # no matching team
        ])

        result = drop_foreign_key_violations(df, self.teams, keys=["team_id"])

        self.assertEqual(result.count(), 1)
        self.assertEqual(result.collect()[0].game_id, "1")

    def test_drop_foreign_key_violations_is_a_no_op_when_all_match(self):
        df = make_game_df([
            ("1", "20162017", "R", "1"),
            ("2", "20162017", "R", "2"),
        ])

        result = drop_foreign_key_violations(df, self.teams, keys=["team_id"])

        self.assertEqual(result.count(), 2)

StatementMeta(, 007ca1de-2eb1-4150-8364-06c3bf8677ac, 102, Finished, Available, Finished, False)

### TestWriteTable

In [52]:
class TestWriteTable(unittest.TestCase):
    """write_table should write via the delta format in overwrite mode
    with schema overwrite enabled, to the given table name."""

    def test_writes_with_expected_options(self):
        mock_df = MagicMock()
        mock_writer = mock_df.write
        mock_writer.format.return_value = mock_writer
        mock_writer.mode.return_value = mock_writer
        mock_writer.option.return_value = mock_writer
        mock_df.count.return_value = 42
        mock_df.columns = ["game_id", "season", "type", "team_id"]

        write_table(mock_df, table_name="silver.game")

        mock_writer.format.assert_called_once_with("delta")
        mock_writer.mode.assert_called_once_with("overwrite")
        mock_writer.option.assert_called_once_with("overwriteSchema", "true")
        mock_writer.saveAsTable.assert_called_once_with("silver.game")

StatementMeta(, 007ca1de-2eb1-4150-8364-06c3bf8677ac, 103, Finished, Available, Finished, False)

### Run all tests

In [53]:
loader = unittest.TestLoader()
suite = unittest.TestSuite()

for test_case in (
    TestLoadTable,
    TestFillNullColumns,
    TestRemoveDuplicates,
    TestDropNoNullColumns,
    TestValidateNoNulls,
    TestValidateColumnValues,    
    TestValidatePrimaryKeys,
    TestForeignKeyFunctions,
    TestWriteTable,
):
    suite.addTests(loader.loadTestsFromTestCase(test_case))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

print("\n========== FULL FAILURE DETAILS ==========")
print(f"Failures: {len(result.failures)}")
print(f"Errors:   {len(result.errors)}")

for test, traceback in result.failures + result.errors:
    print("\n" + "="*60)
    print(test.id())
    print("="*60)
    print(traceback)


StatementMeta(, 007ca1de-2eb1-4150-8364-06c3bf8677ac, 104, Finished, Available, Finished, True)

test_query_references_table_name (__main__.TestLoadTable.test_query_references_table_name) ... ok
test_no_nulls_is_a_no_op (__main__.TestFillNullColumns.test_no_nulls_is_a_no_op) ... ok
test_drops_exact_key_duplicates (__main__.TestRemoveDuplicates.test_drops_exact_key_duplicates) ... ok
test_no_duplicates_is_a_no_op (__main__.TestRemoveDuplicates.test_no_duplicates_is_a_no_op) ... ok
test_drops_rows_with_null_in_required_column (__main__.TestDropNoNullColumns.test_drops_rows_with_null_in_required_column) ... ok
test_no_nulls_is_a_no_op (__main__.TestDropNoNullColumns.test_no_nulls_is_a_no_op) ... ok
test_no_nulls_returns_df_unchanged (__main__.TestValidateNoNulls.test_no_nulls_returns_df_unchanged) ... ok
test_raises_on_null_in_required_column (__main__.TestValidateNoNulls.test_raises_on_null_in_required_column) ... ok
test_all_valid_returns_df_unchanged (__main__.TestValidateColumnValues.test_all_valid_returns_df_unchanged) ... ok
test_raises_on_invalid_values (__main__.TestValidateC

✅ Loaded bronze.game: 1 rows
✅ Loaded bronze.game: 1 rows

========== FULL FAILURE DETAILS ==========
Failures: 0
Errors:   0
